# 課題と解答例：41_love_dynamics_two_person

元Notebook: [../41_love_dynamics_two_person.ipynb](../41_love_dynamics_two_person.ipynb)

## 6. 課題（実装）

1. `classify_linear_system(M)` を作り，トレース，行列式，固有値，分類名を辞書で返す．基準行列と3つの例で実行する．
2. `simulate_linear_system(M, x0, t_end)` を作り，時系列をDataFrameで返す．初期値を複数変えても分類が保たれることを図で確認する．
3. 実部が正の固有値を持つ行列を1つ作り，`classify_linear_system` と数値解の両方で発散を確認する．
4. 外力，飽和，第三者効果はまだ実装せず，どの残差列を記録すれば各拡張を検討できるかをDataFrameの列名として設計する．

## 解答例

1. 分類関数の例である．

   ```python
   def classify_linear_system(M):
       eig = np.linalg.eigvals(M)
       trace = np.trace(M)
       determinant = np.linalg.det(M)
       if determinant < 0:
           label = "saddle"
       elif np.all(eig.real < 0) and np.any(np.abs(eig.imag) > 1e-10):
           label = "stable focus"
       elif np.all(eig.real < 0):
           label = "stable node"
       elif np.any(eig.real > 0):
           label = "unstable"
       else:
           label = "borderline"
       return {"trace": trace, "det": determinant, "eigenvalues": eig, "classification": label}
   ```

2. 時系列をDataFrameにする関数である．

   ```python
   def simulate_linear_system(M, x0, t_end=30):
       t_eval = np.linspace(0, t_end, 600)
       sol = solve_ivp(rhs, (0, t_end), x0, t_eval=t_eval, args=(M,))
       return pd.DataFrame({"t": sol.t, "x1": sol.y[0], "x2": sol.y[1]})
   ```

3. 発散する例である．

   ```python
   M_unstable = np.array([[0.2, 1.0], [-1.0, 0.2]])
   classify_linear_system(M_unstable)
   ```

   固有値の実部が正なので，数値解でも振幅が増える．刻みを細かくしても成長が残るなら，モデルが予測する発散である．

4. 残差記録用の列名例である．

   ```python
   residual_columns = ["time", "observed_x1", "predicted_x1", "residual_x1", "event_label", "third_person_present"]
   ```

   外力は `event_label`，第三者効果は `third_person_present` と残差の対応から検討できる．